# 🧠 Knowledge Graph RAG for Misinformation
### NdabaX 2026 — Hands-on Workshop

Welcome! Over the next hour you'll go from raw text to a **typed, queryable
knowledge graph**, then use it to reason — finishing by cracking a murder mystery
with a KG-RAG detective. Run every cell top-to-bottom.

**Roadmap:** Introduction → RAG & Misinformation → **A.1** Knowledge Representation →
**A.2** Build a Knowledge Base → **B** Use the Graph (murder mystery).

## Setup

In [ ]:
# @title Install dependencies { display-mode: "form" }
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "datasets", "openai", "python-dotenv", "spacy", "pyvis>=0.3.2", "networkx",
    "sentence-transformers>=3.0", "matplotlib", "scipy"], check=True)
subprocess.run([sys.executable, "-m", "spacy", "download", "en_core_web_sm", "-q"], check=True)
print("✅ packages ready")

In [ ]:
# @title Setup { display-mode: "form" }
import sys, os, pathlib

# Locate notebook_src/ whether it sits in the current folder (local, or after
# unzipping into /content) or one level down (e.g. after `git clone`).
_root = pathlib.Path.cwd()
if not (_root / "notebook_src").exists():
    for _p in sorted(_root.glob("*/notebook_src")):
        _root = _p.parent
        break
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# ── API key ──────────────────────────────────────────────────────────────
# Paste your OpenAI key between the quotes for the demo, then DELETE it after.
# Leave blank to fall back to a local .env file.
OPENAI_API_KEY = ""
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

print("✅ notebook_src ready —", _root)

## Part 1 · Introduction

<div style="background:linear-gradient(135deg,#1e3a8a,#3b82f6);border-radius:12px;padding:18px 22px;margin:10px 0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="color:#fff;font-size:20px;font-weight:800;">Part 1 · Introduction</div>
  <div style="color:rgba(255,255,255,.85);font-size:13px;margin-top:3px;">The three ideas behind KG-RAG: LLMs, embeddings and retrieval</div>
</div>

### What are LLMs?

<div style="
  border-left: 4px solid #6366f1; background: #f8fafc;
  border-radius: 0 10px 10px 0; padding: 14px 20px; margin: 4px 0 10px 0;
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">🤖 &nbsp;What are LLMs?</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;">Large Language Models — next-token prediction at scale</div>
</div>

In [ ]:
from notebook_src.visuals import show, concept_llm
show(concept_llm())

### What are Embeddings?

<div style="
  border-left: 4px solid #0d9488; background: #f8fafc;
  border-radius: 0 10px 10px 0; padding: 14px 20px; margin: 4px 0 10px 0;
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">📐 &nbsp;What are Embeddings?</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;">Dense vectors that encode meaning — the engine behind semantic search</div>
</div>

In [ ]:
from notebook_src.visuals import show, concept_embeddings
show(concept_embeddings())

### What is RAG?

<div style="
  border-left: 4px solid #8b5cf6; background: #f8fafc;
  border-radius: 0 10px 10px 0; padding: 14px 20px; margin: 4px 0 10px 0;
  font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">🔗 &nbsp;What is RAG?</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;">Retrieval-Augmented Generation — grounding an LLM in a knowledge base</div>
</div>

In [ ]:
from notebook_src.visuals import show, concept_rag
show(concept_rag())

## Part 2 · RAG Components & Misinformation

<div style="background:linear-gradient(135deg,#c2410c,#f97316);border-radius:12px;padding:18px 22px;margin:10px 0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="color:#fff;font-size:20px;font-weight:800;">Part 2 · RAG Components & Misinformation</div>
  <div style="color:rgba(255,255,255,.85);font-size:13px;margin-top:3px;">How retrieval-augmented generation keeps answers honest</div>
</div>

In [ ]:
from notebook_src.visuals import show
from notebook_src.display import render_rag_components

# A large language model alone answers from fuzzy memory — and will confidently
# invent facts. RAG bolts on a retriever + a knowledge base so answers stay
# grounded, current and citable. Each component is a guard-rail.
show(render_rag_components())

## Hands-on A.1 · Knowledge Representation (Beginner)

<div style="background:linear-gradient(135deg,#1e40af,#2563eb);border-radius:12px;padding:18px 22px;margin:10px 0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="color:#fff;font-size:20px;font-weight:800;">Hands-on A.1 · Knowledge Representation (Beginner)</div>
  <div style="color:rgba(255,255,255,.85);font-size:13px;margin-top:3px;">The pipeline from raw data to a knowledge base — the concepts</div>
</div>

### Data Source

<div style="margin-left:0px;border-left:4px solid #2563eb;background:#eff6ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">📂 &nbsp;Data Source</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Identifying, collecting and citing raw information sources</div>
</div>

In [ ]:
from notebook_src.visuals import show
from notebook_src.display import render_points_card
show(render_points_card("A.1 · Step 1", "Data Source",
    "Where does the knowledge come from?", [
    "Facts live as <b>text</b>: papers, reports, databases, transcripts, web pages.",
    "Prefer <b>authoritative, citable</b> sources — provenance is what makes a KB trustworthy.",
    "Our running example (A.2) uses <b>PubMedQA</b>: real biomedical abstracts with expert Q&A labels.",
]))

### Data / Information Analysis

<div style="margin-left:0px;border-left:4px solid #2563eb;background:#eff6ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🔬 &nbsp;Data / Information Analysis</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Understanding what is actually in the data before extracting</div>
</div>

In [ ]:
from notebook_src.display import render_points_card
show(render_points_card("A.1 · Step 2", "Data / Information Analysis",
    "Know your data before you mine it", [
    "What entities and relationships actually appear? What questions should the KB answer?",
    "Assess <b>quality, coverage and ambiguity</b> — messy input yields a messy graph.",
    "Decide <b>scope</b>: which facts matter, which are noise.",
]))

### Data Processing

<div style="margin-left:0px;border-left:4px solid #2563eb;background:#eff6ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">⚙️ &nbsp;Data Processing</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Turning raw documents into clean, machine-readable units</div>
</div>

In [ ]:
from notebook_src.display import render_points_card
show(render_points_card("A.1 · Step 3", "Data Processing",
    "From messy documents to clean units", [
    "Normalise, segment and structure the text so it can be extracted from reliably.",
    "Two sub-steps follow: <b>pre-processing</b> and <b>dataset creation</b>.",
]))

#### Data Pre-processing

<div style="margin-left:24px;border-left:3px solid #3b82f6;background:#f0f7ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🧹 &nbsp;Data Pre-processing</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Sentence splitting, normalisation, coreference & abbreviation resolution</div>
</div>

In [ ]:
from notebook_src.display import render_points_card
show(render_points_card("A.1 · Step 3a", "Data Pre-processing",
    "Clean the raw text", [
    "Sentence splitting, tokenisation, normalising case & whitespace.",
    "Coreference & abbreviation resolution — “it”, “PCD” → the entity they name.",
    "You'll see this pay off in A.2's <b>entity disambiguation</b> step.",
], grad="135deg,#2563eb,#3b82f6"))

#### Dataset Creation

<div style="margin-left:24px;border-left:3px solid #3b82f6;background:#f0f7ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🗂️ &nbsp;Dataset Creation</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Assembling the exact corpus you will extract from</div>
</div>

In [ ]:
from notebook_src.display import render_points_card
show(render_points_card("A.1 · Step 3b", "Dataset Creation",
    "Assemble the working corpus", [
    "Select the precise passages the KB will be built from.",
    "Pick the store that fits your access pattern → <b>Vector DB vs Graph DB</b> (next).",
], grad="135deg,#2563eb,#3b82f6"))

##### Vector DB vs Graph DB

<div style="margin-left:48px;border-left:3px solid #1d4ed8;background:#eaf2ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🔵🟣 &nbsp;Vector DB vs Graph DB</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Two stores for two jobs — similarity search vs relationship traversal</div>
</div>

In [ ]:
from notebook_src.display import render_vector_vs_graph
show(render_vector_vs_graph())

### Task Evaluation Set

<div style="margin-left:0px;border-left:4px solid #2563eb;background:#eff6ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🎯 &nbsp;Task Evaluation Set</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Held-out questions that measure whether the KB actually works</div>
</div>

In [ ]:
from notebook_src.display import render_points_card
show(render_points_card("A.1 · Step 4", "Task Evaluation Set",
    "How will you know the KB is any good?", [
    "Hold out <b>question–answer pairs</b> the KB should be able to answer.",
    "Measure retrieval quality (recall@k) and answer <b>faithfulness</b> objectively.",
    "In A.2 we even calibrate <b>what a similarity score means</b> using the STS-B benchmark.",
]))

### Knowledge Base Creation

<div style="margin-left:0px;border-left:4px solid #2563eb;background:#eff6ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🏗️ &nbsp;Knowledge Base Creation</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">The end-to-end pipeline — which Hands-on A.2 runs for real</div>
</div>

In [ ]:
from notebook_src.display import render_points_card
show(render_points_card("A.1 · Step 5", "Knowledge Base Creation",
    "Putting it together", [
    "Pipeline: <b>extract</b> triples → <b>type &amp; ground</b> to an ontology → <b>disambiguate</b> → <b>build</b> the graph.",
    "That is exactly what <b>Hands-on A.2</b> does next, on a real abstract.",
    "Result: a typed, queryable knowledge graph — ready for RAG in Hands-on B.",
]))

## Hands-on A.2 · Knowledge Base (Intermediate)

<div style="background:linear-gradient(135deg,#5b21b6,#7c3aed);border-radius:12px;padding:18px 22px;margin:10px 0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="color:#fff;font-size:20px;font-weight:800;">Hands-on A.2 · Knowledge Base (Intermediate)</div>
  <div style="color:rgba(255,255,255,.85);font-size:13px;margin-top:3px;">Build a real knowledge graph from a biomedical abstract, end to end</div>
</div>

### Setup

<div style="margin-left:0px;border-left:4px solid #7c3aed;background:#f5f3ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">🔬 &nbsp;Setup</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Load a real PubMedQA abstract and inspect its structure before we extract knowledge from it.</div>
</div>

In [ ]:
from datasets import load_dataset
from notebook_src.visuals import show
from notebook_src.display import render_abstract

ds  = load_dataset("qiaojin/PubMedQA", "pqa_labeled", split="train",
                   trust_remote_code=True)
ex  = ds[0]

# Metadata (title / authors fetched from NCBI for PMID 21645374)
TITLE   = "Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?"
AUTHORS = "Lord CEN, Wertman JN, Lane S, Gunawardena AHLAN"
JOURNAL = "BMC Plant Biology"
YEAR    = "2011"

# Expose raw fields for downstream cells
question = ex["question"]
contexts = ex["context"]["contexts"]
labels   = ex["context"]["labels"]

show(render_abstract(ex, TITLE, AUTHORS, JOURNAL, YEAR))
print(f"Loaded PMID {ex['pubid']} — {len(contexts)} section(s), "
      f"{sum(len(c) for c in contexts)} chars")


### Information Extraction

<div style="margin-left:0px;border-left:4px solid #7c3aed;background:#f5f3ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">🔬 &nbsp;Information Extraction</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Identifying and pulling structured facts — entities, relations and claims — from text</div>
</div>

In [ ]:
from notebook_src.visuals import show
from notebook_src.display import render_ie_intro

show(render_ie_intro())

#### Ontology-based Information Extraction (OBIE)

<div style="margin-left:24px;border-left:3px solid #6d28d9;background:#f3f0ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🗺️ &nbsp;Ontology-based Information Extraction (OBIE)</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Using a predefined ontology to guide extraction — high precision, closed schema</div>
</div>

In [ ]:
from notebook_src.visuals import show
from notebook_src.display import render_obie_intro

show(render_obie_intro())

In [ ]:
import os
from dotenv import load_dotenv
from notebook_src.visuals import show
from notebook_src.display import render_kg
from notebook_src.extraction import extract_obie

load_dotenv()
API_KEY = os.environ.get("OPENAI_API_KEY", "")

obie_triples = extract_obie(
    contexts=contexts,
    labels=labels,
    ontology_terms=ex["context"]["meshes"],
    api_key=API_KEY,
)

show(render_kg(obie_triples, strategy="OBIE"))
print(f"{len(obie_triples)} OBIE triples extracted")


#### Open-Domain Information Extraction (OpenIE)

<div style="margin-left:24px;border-left:3px solid #6d28d9;background:#f3f0ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🌐 &nbsp;Open-Domain Information Extraction (OpenIE)</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Schema-free extraction across any domain — higher recall, noisier output</div>
</div>

In [ ]:
from notebook_src.visuals import show
from notebook_src.display import render_openie_intro

show(render_openie_intro())

In [ ]:
import os
from dotenv import load_dotenv
from notebook_src.visuals import show
from notebook_src.display import render_kg
from notebook_src.extraction import extract_openie

load_dotenv()
API_KEY = os.environ.get("OPENAI_API_KEY", "")

# Schema-free: the LLM returns EVERY relationship it sees in the text.
openie_triples = extract_openie(
    contexts=contexts,
    labels=labels,
    api_key=API_KEY,
)

# No ontology → colour nodes by abstract section instead of MeSH concept.
show(render_kg(openie_triples, strategy="OpenIE", color_by="section"))
print(f"{len(openie_triples)} OpenIE triples extracted")


### Ontology

<div style="margin-left:0px;border-left:4px solid #7c3aed;background:#f5f3ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">📐 &nbsp;Ontology</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Defining the vocabulary of the KB: node types, edge types and their semantics</div>
</div>

In [ ]:
from notebook_src.visuals import show
from notebook_src.display import render_ontology
from notebook_src.extraction import ONTOLOGY_TYPES

# The 5 MeSH descriptors for this abstract collapse into 4 UMLS semantic types.
show(render_ontology(ONTOLOGY_TYPES))


In [ ]:
import os
from dotenv import load_dotenv
from notebook_src.visuals import show
from notebook_src.display import render_kg_pair, render_grounding
from notebook_src.extraction import ground_to_mesh, filter_by_types, ontology_type_names

load_dotenv()
API_KEY = os.environ.get("OPENAI_API_KEY", "")
meshes  = ex["context"]["meshes"]
allowed = ontology_type_names()

# 1) Ground every entity to the MeSH ontology with a biomedical embedding model
#    (BioLORD). cosine >= 0.8 => same concept => adopt that heading's ontological
#    type. OpenIE also keeps the LLM's inline type guess as a fallback.
obie_typed,   _           = ground_to_mesh(obie_triples,   meshes, threshold=0.8)
openie_typed, openie_rows = ground_to_mesh(openie_triples, meshes, threshold=0.8)

# 2) Show the embedding grounding for the schema-free OpenIE entities.
show(render_grounding(openie_rows, threshold=0.8))

# 3) Filter both graphs to the ontology's types, then compare side by side.
obie_f   = filter_by_types(obie_typed,   allowed)
openie_f = filter_by_types(openie_typed, allowed)
print(f"OBIE  : {len(obie_typed)} -> {len(obie_f)} triples")
print(f"OpenIE: {len(openie_typed)} -> {len(openie_f)} triples  (now the same ontological scope)")

show(render_kg_pair(
    obie_f, openie_f,
    left_title="OBIE", right_title="OpenIE (type-filtered)",
    color_by="type",
))


### Validation

<div style="margin-left:0px;border-left:4px solid #0d9488;background:#f0fdfa;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">✅ &nbsp;Validation</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Back-translate the triples into an abstract and measure how faithfully they preserved the meaning — calibrated against STS-B relatedness levels</div>
</div>

In [ ]:
import os
from dotenv import load_dotenv
from notebook_src.visuals import show
from notebook_src.display import render_abstract_comparison
from notebook_src.validation import back_translate, text_cosine

load_dotenv()
API_KEY = os.environ.get("OPENAI_API_KEY", "")

# Round-trip test: reconstruct the abstract from the OPEN-DOMAIN triples only.
original_abstract = "\n\n".join(contexts)
reconstructed     = back_translate(openie_triples, API_KEY)

# Cosine similarity under the SAME model we calibrate against STS-B below.
sim = text_cosine(original_abstract, reconstructed)
print(f"Back-translation cosine similarity: {sim:.3f}")

show(render_abstract_comparison(original_abstract, reconstructed, sim))

#### Calibrating cosine similarity with STS-B

<div style="margin-left:24px;border-left:3px solid #0d9488;background:#f0fdfa;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:12px;color:#475569;line-height:1.5">A raw cosine number is hard to interpret. <b>STS-B</b> is a benchmark of sentence pairs each rated 0 (unrelated) &rarr; 5 (equivalent) by humans. Running our embedding model over it shows which cosine ranges correspond to which level of relatedness — so we can read our back-translation score against real human judgements.</div>
</div>

In [ ]:
from notebook_src.validation import stsb_cosine_by_category, plot_calibration

# Run the embedding model over STS-B and group cosine by human relatedness level.
cat_cos = stsb_cosine_by_category(split="test")

# Density per level + box-and-whiskers below; mark our back-translation similarity.
level = plot_calibration(cat_cos, highlight=sim, highlight_label="our abstracts")
if level:
    print(f"Our reconstruction (cos={sim:.2f}) sits nearest STS-B level {level[0]} — {level[1]}.")

### Knowledge Representation

<div style="margin-left:0px;border-left:4px solid #7c3aed;background:#f5f3ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">🧠 &nbsp;Knowledge Representation</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Choosing how to encode facts so they are queryable, traversable and maintainable</div>
</div>

In [ ]:
from notebook_src.visuals import show
from notebook_src.display import render_kr_intro

# We have a bag of triples. A knowledge graph is a *property graph*: typed nodes
# and directed, labelled edges, each carrying properties. Four levers get us there.
show(render_kr_intro())

#### Entity Disambiguation

<div style="margin-left:24px;border-left:3px solid #6d28d9;background:#f3f0ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🔗 &nbsp;Entity Disambiguation</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Merging the many surface forms of an entity into a single canonical node</div>
</div>

In [ ]:
from notebook_src.graph import canonicalize_triples
from notebook_src.display import render_disambiguation

# OpenIE emits "mitochondria", "the mitochondria", "Mitochondria" as separate nodes.
# Merge them: cheap lexical normalisation, then BioLORD embedding similarity >= 0.9.
canon_triples, clusters, (n_before, n_after) = canonicalize_triples(openie_typed, threshold=0.9)

show(render_disambiguation(clusters, n_before, n_after, threshold=0.9))
print(f"{n_before} -> {n_after} unique entities  |  {len(canon_triples)} canonical triples")

#### Graph Design

<div style="margin-left:24px;border-left:3px solid #6d28d9;background:#f3f0ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">📊 &nbsp;Graph Design</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Structural decisions that determine how well the graph supports retrieval and reasoning</div>
</div>

In [ ]:
from notebook_src.graph import build_graph
from notebook_src.extraction import filter_by_types, ontology_type_names
from notebook_src.display import render_graph_design

show(render_graph_design())

# Focus the graph on our ontology's types (drop off-schema OpenIE noise), then
# assemble the property graph: directed edges, symmetric-relation flags, temporal
# scopes and provenance are all attached here.
kg_triples = filter_by_types(canon_triples, ontology_type_names())
kg = build_graph(kg_triples)
print(f"Property graph: {kg.number_of_nodes()} nodes, {kg.number_of_edges()} edges "
      f"(disambiguated + ontology-typed)")

##### Properties, Directionality, Temporal & Metadata

<div style="margin-left:48px;border-left:3px solid #6d28d9;background:#f0ecff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🔩 &nbsp;Properties, Directionality, Temporal & Metadata</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Properties: attributes on nodes/edges &middot; Directionality: what A&rarr;B means &middot; Temporal: validity windows &middot; Metadata: provenance, confidence, source</div>
</div>

In [ ]:
from notebook_src.display import render_edge_anatomy

# Dissect one relationship to see every property it carries. Prefer an edge that
# has a temporal scope so all the facets show up at once.
_edges = list(kg.edges(data=True))
_s, _o, _d = next((e for e in _edges if e[2].get("temporal")), _edges[0])
show(render_edge_anatomy({"subject": _s, "object": _o, **_d}))

#### The Consolidated Knowledge Graph

<div style="margin-left:24px;border-left:3px solid #6d28d9;background:#f3f0ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">📈 &nbsp;The Consolidated Knowledge Graph</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">The clean, typed, disambiguated graph — everything assembled into one queryable structure</div>
</div>

In [ ]:
from notebook_src.display import render_kg, render_graph_stats
from notebook_src.graph import graph_stats

# The payoff: one coherent, disambiguated, typed graph — coloured by ontology type.
show(render_kg(kg_triples, strategy="Consolidated KG", color_by="type"))
show(render_graph_stats(graph_stats(kg)))

#### Querying the Knowledge Graph

<div style="margin-left:24px;border-left:3px solid #6d28d9;background:#f3f0ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🔍 &nbsp;Querying the Knowledge Graph</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Traversing relationships to answer questions that flat text cannot</div>
</div>

In [ ]:
from notebook_src.graph import central_entities, neighbours, farthest_path
from notebook_src.display import render_query_panel

hubs  = central_entities(kg, 6)
focus = hubs[0][0]
# the longest chain reachable from the top hub — multi-hop reasoning flat text can't do
path = farthest_path(kg, focus)

show(render_query_panel(focus, neighbours(kg, focus), path))
print("Hubs by degree:", [f"{n} ({d})" for n, d in hubs])

#### Exporting to Neo4j

<div style="margin-left:24px;border-left:3px solid #6d28d9;background:#f3f0ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🗄️ &nbsp;Exporting to Neo4j</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Persisting the property graph as Cypher — ready for a real graph database</div>
</div>

In [ ]:
from notebook_src.graph import to_cypher
from notebook_src.display import render_code_block

# The same graph, serialised as Neo4j CREATE statements.
show(render_code_block(to_cypher(kg, max_edges=25)))

### Key Takeaways

<div style="margin-left:0px;border-left:4px solid #7c3aed;background:#f5f3ff;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">🎓 &nbsp;Key Takeaways</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">The full knowledge-graph generation pipeline, end to end</div>
</div>

In [ ]:
from notebook_src.display import render_takeaways
show(render_takeaways())

## Hands-on B · Use the Graph — A Murder Mystery

<div style="background:linear-gradient(135deg,#7f1d1d,#b91c1c);border-radius:12px;padding:18px 22px;margin:10px 0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="color:#fff;font-size:20px;font-weight:800;">Hands-on B · Use the Graph — A Murder Mystery</div>
  <div style="color:rgba(255,255,255,.85);font-size:13px;margin-top:3px;">Retrieval, prompt engineering and chain-of-thought to crack a case</div>
</div>

### The Case

<div style="margin-left:0px;border-left:4px solid #b91c1c;background:#fef2f2;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🕵️ &nbsp;The Case</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">A guest is dead. Five suspects, one culprit — the graph holds the answer</div>
</div>

In [ ]:
from notebook_src.visuals import show
from notebook_src import mystery as M
from notebook_src.display import render_case_file
show(render_case_file(M.CASE))

### The Mystery Knowledge Graph

<div style="margin-left:0px;border-left:4px solid #b91c1c;background:#fef2f2;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🕸️ &nbsp;The Mystery Knowledge Graph</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Suspects, rooms, weapons and motives as a typed property graph</div>
</div>

In [ ]:
from notebook_src.graph import build_graph
from notebook_src.display import render_kg

case_triples = M.mystery_triples()
mystery_kg   = build_graph(case_triples)

# Suspects (red) linked to rooms (blue), the weapon (grey) and motives (amber).
show(render_kg(case_triples, strategy="Mystery KG", color_by="type",
               group_colours=M.MYSTERY_COLOURS, height="500px"))

### Querying the Graph

<div style="margin-left:0px;border-left:4px solid #b91c1c;background:#fef2f2;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🔗 &nbsp;Querying the Graph</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Traverse the graph to see who connects to the scene of the crime</div>
</div>

In [ ]:
from notebook_src.graph import neighbours, farthest_path
from notebook_src.display import render_query_panel

# Who and what is connected to the Study — the scene of the crime?
scene = "the Study"
show(render_query_panel(scene, neighbours(mystery_kg, scene),
                        farthest_path(mystery_kg, scene)))

### Retrieval with Embeddings

<div style="margin-left:0px;border-left:4px solid #b91c1c;background:#fef2f2;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🔎 &nbsp;Retrieval with Embeddings</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Semantic search finds the handful of facts relevant to a question</div>
</div>

In [ ]:
from notebook_src.display import render_retrieval

question = "Where was Ms. Vivian Scarlett and did she have an alibi?"
hits = M.retrieve(question, k=4)
show(render_retrieval(question, hits))

### Prompt Engineering — Grounded vs Ungrounded

<div style="margin-left:0px;border-left:4px solid #b91c1c;background:#fef2f2;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">✍️ &nbsp;Prompt Engineering — Grounded vs Ungrounded</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">The same question, answered with and without the retrieved facts</div>
</div>

In [ ]:
import os
from notebook_src.display import render_prompt_compare
API_KEY = os.environ.get("OPENAI_API_KEY", "")

q = "What was Ms. Vivian Scarlett's motive, and did she have an alibi?"
context = [h["clue"] for h in M.retrieve("Vivian Scarlett motive alibi partnership Study", 6)]

ungrounded = M.answer(q, [],      API_KEY, grounded=False)   # no graph -> guesses
grounded   = M.answer(q, context, API_KEY, grounded=True)    # KG-RAG -> cites facts
show(render_prompt_compare(q, ungrounded, grounded))

### Chain-of-Thought — Solving the Case

<div style="margin-left:0px;border-left:4px solid #b91c1c;background:#fef2f2;border-radius:0 10px 10px 0;padding:13px 18px;margin-top:4px;margin-bottom:10px;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:14px;font-weight:700;color:#0f172a;">🧩 &nbsp;Chain-of-Thought — Solving the Case</div>
  <div style="font-size:12px;color:#475569;margin-top:3px;line-height:1.5">Reasoning step-by-step over the facts to unmask the murderer</div>
</div>

In [ ]:
from notebook_src.display import render_reasoning
API_KEY = os.environ.get("OPENAI_API_KEY", "")

# Give the model ALL the facts and let it reason: eliminate alibis, find the one
# suspect with motive, means and opportunity.
solution = M.solve(API_KEY)
show(render_reasoning(solution, M.is_correct(solution["culprit"])))
print("Ground truth:", M.CASE["solution"]["culprit"])

### 🎓 Workshop Wrap-up

<div style="border-left:4px solid #7c3aed;background:#f5f3ff;border-radius:0 10px 10px 0;padding:14px 18px;margin:10px 0;font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;">
  <div style="font-size:16px;font-weight:700;color:#0f172a;">🎓 &nbsp;You built a KG-RAG system end to end</div>
  <div style="font-size:12.5px;color:#475569;margin-top:4px;line-height:1.6">
  <b>A.2</b> — extracted, typed, grounded, disambiguated and queried a knowledge graph from raw text.<br>
  <b>B</b> — retrieved facts by meaning, grounded a prompt to stop hallucination, and reasoned step-by-step to solve a case.<br>
  That is the core of using knowledge graphs to make LLMs trustworthy and resistant to misinformation.
  </div>
</div>